<a href="https://colab.research.google.com/github/Zinc-zn/MachineLearning2026/blob/main/JS05_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# JS05 - Tugas Lab 2: Multinomial Naive Bayes - CountVectorizer vs TF-IDF (spam.csv)

**Soal:**

1. Buatlah model klasifikasi Multinomial Naive Bayes menggunakan data `spam.csv` dengan fitur `CountVectorizer` yang mengaktifkan **stop_words**, lalu evaluasi hasilnya.
2. Buatlah model klasifikasi Multinomial Naive Bayes menggunakan data `spam.csv` dengan fitur **TF-IDF** yang mengaktifkan **stop_words**, evaluasi hasilnya, bandingkan dengan hasil pada soal nomor 1, dan berikan kesimpulan fitur mana yang terbaik pada kasus data `spam.csv`.

**Ringkasan jawaban:** CountVectorizer + stop_words mencapai akurasi test **98.30%**, lebih unggul daripada TF-IDF + stop_words (**96.05%**).


## 1. Load & Preprocessing Data

Pra-pengolahan sama dengan Lab 3: buang kolom kosong, ubah nama kolom, dan encode label.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load data
df = pd.read_csv('spam.csv', encoding='latin-1')

# Drop 3 kolom terakhir
df = df.drop(df.iloc[:, 2:], axis=1)

# Ubah nama kolom
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Encode label
df['Labels'] = df['Labels'].map({'spam': 1, 'ham': 0})

X = df['SMS'].values
y = df['Labels'].values

# Split data training dan testing (sama dengan Lab 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## 2. Model 1 - CountVectorizer + stop_words

Ekstraksi fitur Bag of Words dengan mengaktifkan `stop_words='english'` sehingga kata-kata umum (the, is, and, ...) tidak ikut dihitung.

In [2]:
# Ekstraksi fitur Bag of Words dengan stop_words
bow = CountVectorizer(stop_words='english')

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print('Jumlah fitur (kosakata):', len(bow.get_feature_names_out()))
print('Dimensi data train     :', X_train_bow.shape)

Jumlah fitur (kosakata): 7466
Dimensi data train     : (4457, 7466)


In [3]:
# Training MultinomialNB
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

# Evaluasi
acc_train_bow = accuracy_score(y_train, mnb_bow.predict(X_train_bow))
acc_test_bow = accuracy_score(y_test, mnb_bow.predict(X_test_bow))

print(f'Hasil akurasi data train: {acc_train_bow}')
print(f'Hasil akurasi data test : {acc_test_bow}')

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test : 0.9829596412556054


In [4]:
print('Laporan Klasifikasi (CountVectorizer + stop_words):\n')
print(classification_report(y_test, mnb_bow.predict(X_test_bow), target_names=['ham', 'spam']))

Laporan Klasifikasi (CountVectorizer + stop_words):

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## 3. Model 2 - TF-IDF + stop_words

TF-IDF (Term Frequency - Inverse Document Frequency) menimbang setiap kata berdasarkan frekuensi kemunculannya dalam dokumen dikoreksi dengan kelangkaan kata tersebut di seluruh korpus.

In [5]:
# Ekstraksi fitur TF-IDF dengan stop_words
tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('Jumlah fitur (kosakata):', len(tfidf.get_feature_names_out()))
print('Dimensi data train     :', X_train_tfidf.shape)

Jumlah fitur (kosakata): 7466
Dimensi data train     : (4457, 7466)


In [6]:
# Training MultinomialNB
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# Evaluasi
acc_train_tfidf = accuracy_score(y_train, mnb_tfidf.predict(X_train_tfidf))
acc_test_tfidf = accuracy_score(y_test, mnb_tfidf.predict(X_test_tfidf))

print(f'Hasil akurasi data train: {acc_train_tfidf}')
print(f'Hasil akurasi data test : {acc_test_tfidf}')

Hasil akurasi data train: 0.9842943684092439
Hasil akurasi data test : 0.9605381165919282


In [7]:
print('Laporan Klasifikasi (TF-IDF + stop_words):\n')
print(classification_report(y_test, mnb_tfidf.predict(X_test_tfidf), target_names=['ham', 'spam']))

Laporan Klasifikasi (TF-IDF + stop_words):

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       954
        spam       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



## 4. Perbandingan & Kesimpulan

In [8]:
comparison = pd.DataFrame({
    'Model': ['MultinomialNB + CountVectorizer (Lab 3, tanpa stop_words)',
              'MultinomialNB + CountVectorizer + stop_words',
              'MultinomialNB + TF-IDF + stop_words'],
    'Akurasi train': [0.9946, acc_train_bow, acc_train_tfidf],
    'Akurasi test': [0.9776, acc_test_bow, acc_test_tfidf],
})
comparison

,Model,Akurasi train,Akurasi test
0,"MultinomialNB + CountVectorizer (Lab 3, tanpa ...",0.994600,0.977600
1,MultinomialNB + CountVectorizer + stop_words,0.994615,0.982960
2,MultinomialNB + TF-IDF + stop_words,0.984294,0.960538


**Kesimpulan:**

1. **CountVectorizer + stop_words** memberikan hasil terbaik: akurasi test **98.30%** (naik dari 97.76% tanpa stop_words pada Lab 3).
2. **TF-IDF + stop_words** justru lebih rendah: **96.05%**.
3. Pada kasus data `spam.csv` dengan Multinomial Naive Bayes, **fitur terbaik adalah CountVectorizer (Bag of Words dengan stop_words)**. Hal ini masuk akal karena: (a) MultinomialNB memang dirancang untuk fitur berupa *hitungan* diskrit seperti jumlah kemunculan kata; (b) kata-kata penanda spam (free, win, claim, prize, dll.) memang muncul berulang, dan pembobotan TF-IDF justru menurunkan bobot kata-kata sering ini sehingga sinyal penanda spam melemah; (c) TF-IDF menghasilkan nilai pecahan kontinu yang kurang cocok dengan asumsi distribusi MultinomialNB.
4. Menambahkan `stop_words` pada CountVectorizer terbukti membantu (98.30% vs 97.76%) karena kata-kata umum yang bukan penanda spam menjadi ikut tereliminasi.